# Customer purchases — data exploration
The aim is to predict purchase outcomes from age and estimated salary. The first checks cover data quality, class balance and the relationship between the two predictors and purchases.

Missing a purchaser is the main error of interest, so recall is the default tuning metric. Precision and F1 show the cost in extra positive predictions. The data contains no contact costs or purchase values, so these scores cannot be converted into a profit estimate.

This exploration uses the training portion of the default stratified split. The full source is used for schema and quality checks.


## Setup and source schema


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from sklearn.model_selection import train_test_split

locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations for candidate in (location, location / "classification")
     if (candidate / "customer_purchases_dataset/customer_details.csv").is_file()), None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root, classification/, or notebooks/.")
DATA_DIR = PROJECT_ROOT / "customer_purchases_dataset"
SOURCE_PATH = DATA_DIR / "customer_details.csv"
CLEANED_PATH = DATA_DIR / "cleaned_data.csv"
COMMON_ARTIFACTS = PROJECT_ROOT / "artifacts/common"
TARGET = "Purchased"
ID_COLUMN = "RowId"
FEATURE_COLUMNS = ["Age", "EstimatedSalary"]
CLASS_NAMES = ["Did not purchase", "Purchased"]
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [2]:
# Match these settings to a model notebook if you change its split.
RANDOM_STATE = 0
TEST_SIZE = 0.25


In [3]:
raw_data = pd.read_csv(SOURCE_PATH)
print("Source shape:", raw_data.shape)
display(raw_data.head(10))
display(pd.DataFrame({"dtype": raw_data.dtypes.astype(str), "missing_count": raw_data.isna().sum(),
                      "missing_percent": raw_data.isna().mean() * 100, "unique_values": raw_data.nunique()}))
display(raw_data.describe(include="all").T)
print("Exact duplicate source records:", int(raw_data.duplicated().sum()))


Source shape: (400, 3)


,Age,EstimatedSalary,Purchased
0,19,19000,0
1,35,20000,0
2,26,43000,0
3,27,57000,0
4,19,76000,0
5,27,58000,0
6,27,84000,0
7,32,150000,1
8,25,33000,0
9,35,65000,0


,dtype,missing_count,missing_percent,unique_values
Age,int64,0,0.0,43
EstimatedSalary,int64,0,0.0,117
Purchased,int64,0,0.0,2


,count,mean,std,min,25%,50%,75%,max
Age,400.0,37.6550,10.482877,18.0,29.75,37.0,46.0,60.0
EstimatedSalary,400.0,69742.5000,34096.960282,15000.0,43000.00,70000.0,88000.0,150000.0
Purchased,400.0,0.3575,0.479864,0.0,0.00,0.0,1.0,1.0


Exact duplicate source records: 33


## Apply the shared rules for a training-only exploration view
The deterministic cleaning definitions are also included in notebook 02, which writes the common cleaned CSV. Here they create a diagnostic view only. There is no imputation or scaling before the split.


In [4]:
def clean_purchase_data(source):
    """Shared deterministic cleaning; no imputation or learned scaling here."""
    required = [*FEATURE_COLUMNS, TARGET]
    missing = set(required) - set(source.columns)
    if missing:
        raise ValueError(f"Missing required source columns: {sorted(missing)}")
    data = source[required].copy()
    data.insert(0, ID_COLUMN, np.arange(1, len(source) + 1))
    parsing_counts = {}
    for column in required:
        original = data[column].copy()
        data[column] = pd.to_numeric(data[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
        parsing_counts[column] = int((original.notna() & data[column].isna()).sum())
    valid_label = data[TARGET].isin([0, 1])
    rejected_labels = data.loc[~valid_label].copy()
    rejected_labels["reason"] = "Missing, nonfinite or non-binary Purchased label"
    data = data.loc[valid_label].copy()
    data[FEATURE_COLUMNS] = data[FEATURE_COLUMNS].astype(float)
    invalid_age = data.Age.notna() & (data.Age <= 0)
    invalid_salary = data.EstimatedSalary.notna() & (data.EstimatedSalary < 0)
    data.loc[invalid_age, "Age"] = np.nan
    data.loc[invalid_salary, "EstimatedSalary"] = np.nan
    no_features = data[FEATURE_COLUMNS].isna().all(axis=1)
    rejected_features = data.loc[no_features].copy()
    rejected_features["reason"] = "Both predictors missing or invalid"
    data = data.loc[~no_features].copy()
    # There is no customer identifier in this source. Deduplicate equivalent
    # labeled records conservatively; retain different outcomes for a profile.
    duplicate = data.duplicated(subset=required, keep="first")
    duplicate_rows = data.loc[duplicate].copy()
    duplicate_rows["reason"] = "Repeated equivalent labeled record"
    data = data.loc[~duplicate].copy()
    data[TARGET] = data[TARGET].astype(int)
    data = data.reset_index(drop=True)
    if set(data[TARGET].unique()) != {0, 1} or data[TARGET].value_counts().min() < 4:
        raise ValueError("Need both classes with at least four usable rows per class")
    if data[FEATURE_COLUMNS].isna().all().any():
        raise ValueError("A predictor is entirely missing; cannot fit a training median")
    audit = {
        "source_rows": len(source), "invalid_label_rows_excluded": len(rejected_labels),
        "all_predictors_missing_rows_excluded": len(rejected_features),
        "duplicate_labeled_rows_excluded": int(duplicate.sum()), "cleaned_rows": len(data),
        "invalid_age_values_set_missing": int(invalid_age.sum()),
        "invalid_salary_values_set_missing": int(invalid_salary.sum()),
        "non_numeric_or_nonfinite_values": parsing_counts,
        "remaining_missing_predictors": data[FEATURE_COLUMNS].isna().sum().to_dict(),
        "class_counts": data[TARGET].value_counts().sort_index().to_dict(),
    }
    excluded = pd.concat([rejected_labels, rejected_features, duplicate_rows], ignore_index=True)
    return data, audit, excluded


In [5]:
cleaned_view, cleaning_audit, excluded = clean_purchase_data(raw_data)
display(pd.Series(cleaning_audit, name="cleaning_audit").to_frame())
training_view, testing_view = train_test_split(cleaned_view, test_size=TEST_SIZE,
                                              random_state=RANDOM_STATE, stratify=cleaned_view[TARGET])
eda = training_view.copy()
print("Training rows for exploration:", len(eda), "reserved test rows:", len(testing_view))


,cleaning_audit
source_rows,400
invalid_label_rows_excluded,0
all_predictors_missing_rows_excluded,0
duplicate_labeled_rows_excluded,33
cleaned_rows,367
invalid_age_values_set_missing,0
invalid_salary_values_set_missing,0
non_numeric_or_nonfinite_values,"{'Age': 0, 'EstimatedSalary': 0, 'Purchased': 0}"
remaining_missing_predictors,"{'Age': 0, 'EstimatedSalary': 0}"
class_counts,"{0: 229, 1: 138}"


Training rows for exploration: 275 reserved test rows: 92


## Purchase balance
The cleaned training split contains 172 non-purchases and 103 purchases. Non-purchases are the larger class, so accuracy needs to be read alongside recall and precision. `Purchased = 1` is the positive class.


In [6]:
balance = eda[TARGET].value_counts().sort_index().rename_axis(TARGET).reset_index(name="customers")
balance["percent"] = balance.customers / len(eda) * 100
display(balance)
px.bar(balance, x=TARGET, y="customers", text="percent", title="Training purchase-class balance").show()


,Purchased,customers,percent
0,0,172,62.545455
1,1,103,37.454545


## Feature distributions and unusual values
Age and salary have very different scales. Valid high salaries are retained; an IQR flag alone is not a reason to remove them. Scaling is fitted within each model's training folds.


In [7]:
display(eda[FEATURE_COLUMNS].describe().T.assign(skewness=eda[FEATURE_COLUMNS].skew()))
for column in FEATURE_COLUMNS:
    px.histogram(eda, x=column, color=TARGET, nbins=25, barmode="overlay", title=f"Training {column} by purchase outcome").show()
    px.box(eda, x=TARGET, y=column, title=f"Training {column}: purchase groups").show()
q1, q3 = eda[FEATURE_COLUMNS].quantile(.25), eda[FEATURE_COLUMNS].quantile(.75)
iqr_flags = (eda[FEATURE_COLUMNS] < q1 - 1.5 * (q3 - q1)) | (eda[FEATURE_COLUMNS] > q3 + 1.5 * (q3 - q1))
display(iqr_flags.sum().to_frame("iqr_review_flags"))
display(eda.loc[iqr_flags.any(axis=1)])


,count,mean,std,min,25%,50%,75%,max,skewness
Age,275.0,37.189091,10.528059,18.0,29.0,36.0,46.0,60.0,0.225945
EstimatedSalary,275.0,71120.000000,34929.597597,15000.0,45000.0,71000.0,88000.0,150000.0,0.444472


,iqr_review_flags
Age,0
EstimatedSalary,0


,RowId,Age,EstimatedSalary,Purchased


## Age, salary and purchase outcomes
Purchasers in this training split have a median age of 47 and median estimated salary of 97,000, compared with 32.5 and 61,000 for non-purchasers. The groups still overlap, so neither feature provides a clean cutoff on its own.


In [8]:
px.scatter(eda, x="Age", y="EstimatedSalary", color=eda[TARGET].map({0: "Did not purchase", 1: "Purchased"}),
           hover_data=[ID_COLUMN], title="Training customers: age, salary and purchase outcome").show()
px.imshow(eda[[*FEATURE_COLUMNS, TARGET]].corr(), zmin=-1, zmax=1, color_continuous_scale="RdBu_r", title="Training correlations; RowId excluded").show()
display(eda.groupby(TARGET)[FEATURE_COLUMNS].agg(["count", "median", "mean", "std"]))


Age                             EstimatedSalary           \
          count median       mean       std           count   median   
Purchased                                                              
0           172   32.5  32.046512  7.920522             172  61000.0   
1           103   47.0  45.776699  8.573708             103  97000.0   

                                       
                   mean           std  
Purchased                              
0          59802.325581  23788.699350  
1          90019.417476  41857.597193

## Findings and next steps
The source has 400 rows, no missing values and 33 repeated labeled records. Removing these repeats leaves 367 rows. The training view shows differences in both age and salary between purchase groups, with enough overlap to test both linear and nonlinear boundaries.

Run 02 to save the cleaned data, then run a classifier notebook. Hyperparameter selection uses training CV; the test split is reserved for final scoring.
